In [62]:
# run autoreload
get_ipython().run_line_magic('load_ext', 'autoreload')
get_ipython().run_line_magic('autoreload', '2')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import json
import sys
import importlib
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "src" / "zjet_corrections").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str((repo_root / "src").resolve()))

import zjet_corrections.notebook_utils as nbutils
importlib.reload(nbutils)

# -------------------- Persistence --------------------
CONFIG_FILE = Path(".analysis_widget_config.json")

DEFAULTS = {
    "casa": True,
    "test": False,
    "data": False,
    "useDefault": False,
    "mode": "validation",
    "era": "2016",
    "dataset": "pythia",
    "chunksize": 50000,
    "chunksize_test": 200000,
    "group_mode": "all_in_one",
    "prependstr": "root://xcache/",
    "systematic_profile": "all_syst",
}


def load_config():
    """Load last saved config if available, otherwise return defaults."""
    if CONFIG_FILE.exists():
        try:
            with open(CONFIG_FILE, "r") as f:
                loaded = json.load(f)

            config = DEFAULTS.copy()
            config.update(loaded)
            return config

        except Exception as e:
            print(f"Warning: could not read {CONFIG_FILE}: {e}")
            print("Falling back to defaults.")
    return DEFAULTS.copy()



def save_config(config):
    """Save current config to disk."""
    try:
        with open(CONFIG_FILE, "w") as f:
            json.dump(config, f, indent=2)
    except Exception as e:
        print(f"Warning: could not save config: {e}")



def apply_config_to_widgets(config):
    """Push config values into widgets."""
    casa_w.value = config["casa"]
    test_w.value = config["test"]
    data_w.value = config["data"]
    useDefault_w.value = config["useDefault"]
    mode_w.value = config["mode"]
    era_w.value = config["era"]
    dataset_w.value = config["dataset"]
    chunksize_w.value = config["chunksize"]
    chunksize_test_w.value = config["chunksize_test"]
    group_mode_w.value = config["group_mode"]
    prependstr_w.value = config["prependstr"]
    systematic_profile_w.value = config["systematic_profile"]



def get_config_from_widgets():
    """Pull current widget values into a config dict."""
    return {
        "casa": casa_w.value,
        "test": test_w.value,
        "data": data_w.value,
        "useDefault": useDefault_w.value,
        "mode": mode_w.value,
        "era": era_w.value,
        "dataset": dataset_w.value,
        "chunksize": chunksize_w.value,
        "chunksize_test": chunksize_test_w.value,
        "group_mode": group_mode_w.value,
        "prependstr": prependstr_w.value,
        "systematic_profile": systematic_profile_w.value,
    }



def apply_widget_values(save=False, show_output=True):
    """
    Read current widget values and apply them to notebook variables.
    Optionally save them and/or print them.
    """
    global casa, test, mode, era, data, dataset
    global chunksize, chunksize_test, useDefault
    global group_mode, prependstr, systematic_profile
    global systematics_list, jet_systematics_list

    casa = casa_w.value
    test = test_w.value
    mode = mode_w.value
    era = era_w.value
    data = data_w.value
    dataset = dataset_w.value
    chunksize = chunksize_w.value
    chunksize_test = chunksize_test_w.value
    useDefault = useDefault_w.value
    group_mode = group_mode_w.value
    prependstr = prependstr_w.value
    systematic_profile = systematic_profile_w.value
    systematics_list, jet_systematics_list = nbutils.resolve_systematics(systematic_profile)

    if save:
        save_config(get_config_from_widgets())

    if show_output:
        with output:
            output.clear_output()
            print("Applied settings:")
            print(f"casa = {casa}")
            print(f"test = {test}")
            print(f"mode = {mode}")
            print(f"era = {era}")
            print(f"data = {data}")
            print(f"dataset = {dataset}")
            print(f"systematic_profile = {systematic_profile}")
            print(f"systematics_list = {systematics_list}")
            print(f"jet_systematics_list = {jet_systematics_list}")
            print(f"chunksize = {chunksize}")
            print(f"chunksize_test = {chunksize_test}")
            print(f"useDefault = {useDefault}")
            print(f"group_mode = {group_mode}")
            print(f"prependstr = {prependstr}")
            print("output_dir = outputs/")
            if save:
                print(f"\nSaved to: {CONFIG_FILE}")


# Load last saved config (or defaults if none exists)
CONFIG = load_config()

DATASET_OPTIONS = nbutils.DATASET_OPTIONS
ERA_OPTIONS = nbutils.ERA_OPTIONS
MODE_OPTIONS = nbutils.MODE_OPTIONS
SYSTEMATIC_PROFILE_OPTIONS = nbutils.SYSTEMATIC_PROFILE_OPTIONS

if CONFIG["dataset"] not in DATASET_OPTIONS:
    CONFIG["dataset"] = DEFAULTS["dataset"]
if CONFIG["era"] not in ERA_OPTIONS:
    CONFIG["era"] = DEFAULTS["era"]
if CONFIG["mode"] not in MODE_OPTIONS:
    CONFIG["mode"] = DEFAULTS["mode"]
if CONFIG.get("systematic_profile") not in SYSTEMATIC_PROFILE_OPTIONS:
    CONFIG["systematic_profile"] = DEFAULTS["systematic_profile"]

# -------------------- Widgets --------------------
casa_w = widgets.Checkbox(value=CONFIG["casa"], description="casa")
test_w = widgets.Checkbox(value=CONFIG["test"], description="test")
data_w = widgets.Checkbox(value=CONFIG["data"], description="data")
useDefault_w = widgets.Checkbox(value=CONFIG["useDefault"], description="useDefault")

mode_w = widgets.Dropdown(
    options=MODE_OPTIONS,
    value=CONFIG["mode"],
    description="mode"
)

era_w = widgets.Dropdown(
    options=ERA_OPTIONS,
    value=CONFIG["era"],
    description="era"
)

dataset_w = widgets.Dropdown(
    options=DATASET_OPTIONS,
    value=CONFIG["dataset"],
    description="dataset"
)

systematic_profile_w = widgets.Dropdown(
    options=SYSTEMATIC_PROFILE_OPTIONS,
    value=CONFIG["systematic_profile"],
    description="syst profile"
)

chunksize_w = widgets.IntText(
    value=CONFIG["chunksize"],
    description="chunksize"
)

chunksize_test_w = widgets.IntText(
    value=CONFIG["chunksize_test"],
    description="test chunk"
)

group_mode_w = widgets.Dropdown(
    options=["all_in_one", "per_group"],
    value=CONFIG["group_mode"],
    description="group_mode"
)

prependstr_w = widgets.Text(
    value=CONFIG["prependstr"],
    description="prependstr"
)

run_button = widgets.Button(
    description="Apply settings",
    button_style="success"
)

reset_button = widgets.Button(
    description="Reset to defaults",
    button_style="warning"
)

output = widgets.Output()

# -------------------- Layout --------------------
ui = widgets.VBox([
    widgets.HTML("<h3>Analysis configuration</h3>"),
    widgets.HBox([casa_w, test_w, data_w, useDefault_w]),
    mode_w,
    era_w,
    dataset_w,
    systematic_profile_w,
    chunksize_w,
    chunksize_test_w,
    group_mode_w,
    prependstr_w,
    widgets.HBox([run_button, reset_button]),
    output
])

display(ui)

# -------------------- Button actions --------------------
def on_run_clicked(b):
    apply_widget_values(save=True, show_output=True)



def on_reset_clicked(b):
    apply_config_to_widgets(DEFAULTS)
    apply_widget_values(save=True, show_output=True)

    with output:
        output.clear_output()
        print("Reset all widget values to defaults and applied them.")
        print(f"Saved defaults to: {CONFIG_FILE}")
        print(f"casa = {casa}")
        print(f"test = {test}")
        print(f"mode = {mode}")
        print(f"era = {era}")
        print(f"data = {data}")
        print(f"dataset = {dataset}")
        print(f"systematic_profile = {systematic_profile}")
        print(f"systematics_list = {systematics_list}")
        print(f"jet_systematics_list = {jet_systematics_list}")
        print(f"chunksize = {chunksize}")
        print(f"chunksize_test = {chunksize_test}")
        print(f"useDefault = {useDefault}")
        print(f"group_mode = {group_mode}")
        print(f"prependstr = {prependstr}")
        print("output_dir = outputs/")


run_button.on_click(on_run_clicked)
reset_button.on_click(on_reset_clicked)

# -------------------- Auto-apply on cell execution --------------------
apply_widget_values(save=False, show_output=True)


In [64]:
paths = nbutils.get_analysis_paths(repo_root)
HT_BINS = nbutils.HT_BINS

SAMPLES_DATA_DIR = paths.samples_data_dir
SAMPLES_MC_DIR = paths.samples_mc_dir
SAMPLES_BKG_DIR = paths.samples_bkg_dir
SAMPLES_MC_LOCAL_DIR = paths.samples_mc_local_dir

samplePath = nbutils.SamplePath(era)

# In[4]:  -------------------- Imports (do once) --------------------
import os
import time
import pickle
import importlib

import numpy as np
import awkward as ak
import uproot

import coffea
from coffea.nanoevents import NanoAODSchema
from coffea import processor
from IPython.display import Audio

importlib.reload(nbutils)
import dask
dask.config.set({
    "distributed.logging.distributed": "error",
    "distributed.logging.bokeh": "error",
    "distributed.logging.tornado": "error",
})


In [65]:
# In[5]:  -------------------- Helpers --------------------
NanoAODSchema.warn_missing_crossrefs = False

format_time = nbutils.format_time
iter_groups = nbutils.iter_groups
build_fileset_from_txts = nbutils.build_fileset_from_txts
build_backgrounds_fileset = nbutils.build_backgrounds_fileset
build_local_pythia_fileset = nbutils.build_local_pythia_fileset
make_runner = nbutils.make_runner
ensure_client = nbutils.ensure_client
upload_package_if_casa = nbutils.upload_package_if_casa
run_once = nbutils.run_once
save_output = nbutils.save_output
make_output_filename = nbutils.make_output_filename
get_group_tag = nbutils.get_group_tag
ST_FILES = nbutils.ST_FILES


In [66]:
# In[6]:  -------------------- Create client --------------------
client = ensure_client(casa=casa, test=test, useDefault = useDefault)
upload_package_if_casa(client, casa=casa)

Running locally with 1-2 files (test=True)


In [67]:
# In[7]:  -------------------- Build fileset(s), run, and save immediately --------------------
outs = []  # keep multiple outputs if you run multiple groups



def run_save_append(
    fileset,
    i,
    *,
    client=None,
    test=False,
    data=False,
    chunksize=None,
    chunksize_test=None,
):
    out_i = run_once(
        fileset,
        client=client,
        test=test,
        data=data,
        mode=mode,
        systematic_profile=systematic_profile,
        chunksize=chunksize,
        chunksize_test=chunksize_test,
    )

    tag = get_group_tag(i, era, group_mode)
    fout = make_output_filename(
        data=data,
        dataset=dataset,
        tag=tag,
        mode=mode,
        test=test,
    )

    save_output(out_i, fout)
    print(f"[{i+1}] Saved: {fout}")

    outs.append(out_i)
    return out_i


if data:
    for i, group in enumerate(iter_groups(samplePath.data, group_mode)):
        fileset = build_fileset_from_txts(
            group,
            SAMPLES_DATA_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            i,
            client=client,
            test=test,
            data=True,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

else:
    if dataset == "pythia":
        for i, group in enumerate(iter_groups(samplePath.pythia, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=True,
                ht_bins=HT_BINS,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "pythia_local":
        fileset = build_local_pythia_fileset(SAMPLES_MC_LOCAL_DIR, era)
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "pythia2":
        fileset = build_fileset_from_txts(
            ["inclusive_UL16NanoAODv9.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "herwig":
        for i, group in enumerate(iter_groups(samplePath.herwig, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=False,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "powheg":
        fileset = build_fileset_from_txts(
            ["powheg_UL18NanoAODv9_inclusive.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "st":
        fileset = build_fileset_from_txts(
            ST_FILES,
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "backgrounds":
        fileset = build_backgrounds_fileset(
            SAMPLES_BKG_DIR,
            prependstr,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    else:
        print(f"Dataset is {dataset} and it is not in the list")


# In[8]:  -------------------- Choose what to keep in `out` --------------------
out = outs[-1] if outs else None


# In[10]:  -------------------- Analysis / plotting zone --------------------
# Keep plotting down here so the run block stays clean.
# (Your existing plotting cells can remain, just moved below this line.)

print(f"Number of group outputs: {len(outs)}")

if client is not None:
    client.close()


Running over: ['pythia_UL18NanoAODv9_HT-400to600'] 
Running over test files: ['pythia_UL18NanoAODv9_HT-400to600']
[DEBUG] Registered Histograms dict_keys(['jk_response_matrix_u', 'jk_response_matrix_g', 'sumw', 'nev', 'cutflow'])


Output()

[DEBUG] Systematics ['nominal']
[DEBUG] Current Mode jk_mc
[INFO] Starting processing for dataset: pythia_UL18NanoAODv9_HT-400to600 and file: /mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/tests/samples_mc/files/store/mc/RunIISummer20UL18NanoAODv9/DYJetsToLL_M-50_HT-400to600_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/270000/90EAC274-6CB4-BB42-8F60-BF38CBBC26DC.root
[DEBUG] Total events in chunk: 32720
[INFO] year: RunIISummer20UL18NanoAODv9, ht_bin: HT-400to600, herwig: False
[DEBUG] Weights initialized
ht bin HT-400to600
[INFO] Applying MET filters
[DEBUG] PU weights (nom, up, down) : [1.00662441 0.88210136 1.02818737 0.91235547 1.01166006 1.03718
 1.07108805 0.99286314 0.98016382 0.99286314]
[DEBUG] pdf weights (nom, up, down) : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[DEBUG] L1 prefiring weights (nom, up, down) : [0.995, 1, 1, 0.996, 1, 1, 1, 0.995, 0.999, 1]
[DEBUG] Q2 weights (nom, up, down) : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[INFO] Entering

/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/awkward/_connect/_numpy.py:197: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(
/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/awkward/_connect/_numpy.py:197: RuntimeWarning: invalid value encountered in divide
  result = getattr(ufunc, method)(


/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/awkward/_connect/_numpy.py:197: RuntimeWarning: divide by zero encountered in divide
  result = getattr(ufunc, method)(


...

Using TAG AK4PFPuppi


[DEBUG] Jet Corrections Applied
[DEBUG] Available Jet systematics ['nominal']
[DEBUG] Processing jet systematic: nominal
[DEBUG] FatJet pt before correction: [[240], [214], [297, 189], [246, 196], ... 299, 256], [251, 184], [346], [339, 306]]
How many none in Fatjet.mass before processing inside jmrnom 3972
Genmass inside jmrsf [[], [43.1], [40.7], [55.8, 33.3], [38.1, ... 36, 97.6], [48.1], [46.9], [109, 59.7]]
How many none in Fatjet.matched_gen inside jmrnom 0
[[], [0.99], [0.921], [1.01, 1.03], ... 0.997], [0.996], [1.01], [1.01, 1.01]]
How many none in Fatjet inside jmrnom 0
How many none in Fatjet.mass that is being returned inside jmrnom 3974
[DEBUG] FatJet pt after correction: [[240], [213], [299, 189], [248], [322, ... [299, 255], [250], [351], [339, 306]]
[DEBUG] Sum of this sel is 2897
[DEBUG] Len? 2897  
[DEBUG] Padded Electron/Muon collections to minimum length 2 per event


[INFO] Lepton weights added
[DEBUG] Total reco events passing all selection: 2241
[DEBUG] Total gen events passing all selection: 3208
[DEBUG] Total events passing both reco and gen selections: 2122
[DEBUG] Total reco events (ee channel) passing all selection: 827
[DEBUG] Total reco events (mm channel) passing all selection: 1414
[DEBUG] Weights sample: [0.90249344 0.92886713 0.91561802 0.95384081 0.92042657 0.98672292
 0.91346933 0.91235884 0.96871894 0.81802335]
Now doing mm
JET SYST nominal
[DEBUG] list of systematics ['nominal']
[DEBUG] Processing systematic nominal
Fatjet y  [-0.299, -0.866, 1.15, -0.673, -0.9, 0.838, ... -0.0507, 0.81, -1.77, 0.605, -0.0967]
Fatjet eta  [-0.308, -0.875, 1.16, -0.689, -0.915, ... -0.0517, 0.815, -1.88, 0.609, -0.0971]
[DEBUG] Len of ptreco 1414 mreco 1414 syst nominal channel mm dataset pythia_UL18NanoAODv9_HT-400to600
[DEBUG] ptreco sample [210, 347, 291, 298, 471, 319, 517, 478, 519, 365]
[DEBUG] mreco sample [54.5, 54, 50, 70.4, 96.7, 71.5, 160

[INFO] Scaled jk_response_matrix_u for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled jk_response_matrix_g for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
Done. time taken 2m 56s
Output written to outputs/mass_jk_mc_pythia_local_2018_TEST.pkl with size 8.4 MB
[1] Saved: outputs/mass_jk_mc_pythia_local_2018_TEST.pkl
Number of group outputs: 1


/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/src/zjet_corrections/hist_utils.py:23: UserWarning: Please use 'Weight()' instead of 'Weight'
  hnew = hist.Hist(
/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/hist/basehist.py:549: UserWarning: List indexing selection is experimental. Removed bins are not placed in overflow.
  return super().__getitem__(self._index_transform(index))


In [68]:
for key in out:
    print(key)

jk_response_matrix_u
jk_response_matrix_g
sumw
nev
cutflow


In [55]:
# In[7]:  -------------------- Build fileset(s), run, and save immediately --------------------
outs = []  # keep multiple outputs if you run multiple groups



def run_save_append(
    fileset,
    i,
    *,
    client=None,
    test=False,
    data=False,
    chunksize=None,
    chunksize_test=None,
):
    out_i = run_once(
        fileset,
        client=client,
        test=test,
        data=data,
        mode=mode,
        systematic_profile=systematic_profile,
        chunksize=chunksize,
        chunksize_test=chunksize_test,
    )

    tag = get_group_tag(i, era, group_mode)
    fout = make_output_filename(
        data=data,
        dataset=dataset,
        tag=tag,
        mode=mode,
        test=test,
    )

    save_output(out_i, fout)
    print(f"[{i+1}] Saved: {fout}")

    outs.append(out_i)
    return out_i


if data:
    for i, group in enumerate(iter_groups(samplePath.data, group_mode)):
        fileset = build_fileset_from_txts(
            group,
            SAMPLES_DATA_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            i,
            client=client,
            test=test,
            data=True,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

else:
    if dataset == "pythia":
        for i, group in enumerate(iter_groups(samplePath.pythia, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=True,
                ht_bins=HT_BINS,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "pythia_local":
        fileset = build_local_pythia_fileset(SAMPLES_MC_LOCAL_DIR, era)
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "pythia2":
        fileset = build_fileset_from_txts(
            ["inclusive_UL16NanoAODv9.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "herwig":
        for i, group in enumerate(iter_groups(samplePath.herwig, group_mode)):
            fileset = build_fileset_from_txts(
                group,
                SAMPLES_MC_DIR,
                prependstr,
                split_ht=False,
            )
            run_save_append(
                fileset,
                i,
                client=client,
                test=test,
                data=False,
                chunksize=chunksize,
                chunksize_test=chunksize_test,
            )

    elif dataset == "powheg":
        fileset = build_fileset_from_txts(
            ["powheg_UL18NanoAODv9_inclusive.txt"],
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "st":
        fileset = build_fileset_from_txts(
            ST_FILES,
            SAMPLES_MC_DIR,
            prependstr,
            split_ht=False,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    elif dataset == "backgrounds":
        fileset = build_backgrounds_fileset(
            SAMPLES_BKG_DIR,
            prependstr,
        )
        run_save_append(
            fileset,
            0,
            client=client,
            test=test,
            data=False,
            chunksize=chunksize,
            chunksize_test=chunksize_test,
        )

    else:
        print(f"Dataset is {dataset} and it is not in the list")


# In[8]:  -------------------- Choose what to keep in `out` --------------------
out = outs[-1] if outs else None


# In[10]:  -------------------- Analysis / plotting zone --------------------
# Keep plotting down here so the run block stays clean.
# (Your existing plotting cells can remain, just moved below this line.)

print(f"Number of group outputs: {len(outs)}")

if client is not None:
    client.close()


Running over: ['pythia_UL18NanoAODv9_HT-400to600'] 
Running over test files: ['pythia_UL18NanoAODv9_HT-400to600']
[DEBUG] Registered Histograms dict_keys(['ptjet_mjet_u_reco', 'ptjet_mjet_g_reco', 'response_matrix_u', 'response_matrix_g', 'ptjet_mjet_u_gen', 'ptjet_mjet_g_gen', 'ptz_mz_reco', 'sumw', 'nev', 'cutflow'])


Output()

[DEBUG] Systematics ['nominal']
[DEBUG] Current Mode minimal
[INFO] Starting processing for dataset: pythia_UL18NanoAODv9_HT-400to600 and file: /mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/tests/samples_mc/files/store/mc/RunIISummer20UL18NanoAODv9/DYJetsToLL_M-50_HT-400to600_TuneCP5_PSweights_13TeV-madgraphMLM-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/270000/90EAC274-6CB4-BB42-8F60-BF38CBBC26DC.root
[DEBUG] Total events in chunk: 32720
[DEBUG] Jackknife resampling not enabled, processing all events together.
[INFO] year: RunIISummer20UL18NanoAODv9, ht_bin: HT-400to600, herwig: False
[DEBUG] Weights initialized
ht bin HT-400to600
[INFO] Applying MET filters
[DEBUG] PU weights (nom, up, down) : [0.99084239 1.00662441 0.88210136 1.02818737 0.91235547 1.01166006
 1.03718    1.07108805 0.99286314 0.98016382]
[DEBUG] pdf weights (nom, up, down) : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[DEBUG] L1 prefiring weights (nom, up, down) : [1, 0.995, 1, 1, 0.996, 1, 1, 1, 0.995, 0.999]
[DEB

/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/awkward/_connect/_numpy.py:197: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(
/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/awkward/_connect/_numpy.py:197: RuntimeWarning: invalid value encountered in divide
  result = getattr(ufunc, method)(
/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/awkward/_connect/_numpy.py:197: RuntimeWarning: divide by zero encountered in divide
  result = getattr(ufunc, method)(


[INFO] Entering RECO selection
[DEBUG] MET Filter applied
[DEBUG] Leptons Selected
[DEBUG] Z Object Created
Using TAG AK8PFPuppi


...

[DEBUG] Sum of this sel is 3196
[DEBUG] Len? 3196  
[DEBUG] Padded Electron/Muon collections to minimum length 2 per event
[INFO] Lepton weights added
[DEBUG] Total reco events passing all selection: 2470
[DEBUG] Total gen events passing all selection: 3536
[DEBUG] Total events passing both reco and gen selections: 2344
[DEBUG] Total reco events (ee channel) passing all selection: 908
[DEBUG] Total reco events (mm channel) passing all selection: 1562
[DEBUG] Weights sample: [0.94650767 0.90249344 0.92886713 0.91561802 0.95384081 0.92042657
 0.98672292 0.91346933 0.91235884 0.96871894]
Now doing mm
JET SYST nominal
[DEBUG] list of systematics ['nominal']
[DEBUG] Processing systematic nominal


/mnt/8A04C21E04C20CDF/wsLinux/zjet_corrections/.venv/lib/python3.10/site-packages/awkward/_connect/_numpy.py:197: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(


Fatjet y  [0.704, -0.299, -0.866, 1.15, -0.673, -0.9, ... 0.81, -1.77, 0.605, -0.0967, -1.87]
Fatjet eta  [0.707, -0.308, -0.875, 1.16, -0.689, ... 0.815, -1.88, 0.609, -0.0971, -1.89]
[DEBUG] Len of ptreco 1562 mreco 1562 syst nominal channel mm dataset pythia_UL18NanoAODv9_HT-400to600
[DEBUG] ptreco sample [249, 210, 347, 291, 298, 471, 319, 517, 478, 275]
[DEBUG] mreco sample [24.8, 54.5, 54, 50, 70.4, 96.7, 71.5, 160, 48.2, 8.41]
[DEBUG] mreco_g sample [3.02, 55.8, 3.56, 2.25, 6.38, 95.1, 40.6, 169, 0.734, 1.74]
Now doing ee
JET SYST nominal
[DEBUG] list of systematics ['nominal']
[DEBUG] Processing systematic nominal
Fatjet y  [-0.527, 1.44, 1.36, 0.402, 1.01, 0.571, ... 0.637, -0.912, 0.166, -1.51, -2.03]
Fatjet eta  [-0.529, 1.49, 1.37, 0.407, 1.06, 0.59, ... 1.09, 0.654, -0.92, 0.167, -1.51, -2.03]


[DEBUG] Len of ptreco 908 mreco 908 syst nominal channel ee dataset pythia_UL18NanoAODv9_HT-400to600
[DEBUG] ptreco sample [536, 206, 524, 241, 315, 212, 342, 247, 209, 264]
[DEBUG] mreco sample [54, 69.3, 53.5, 41.5, 118, 58.6, 122, 42.8, 59.5, 74.3]
[DEBUG] mreco_g sample [5.87, 70.6, 28, 37.8, 102, 49.2, 126, 13.2, 26.8, 71.8]


[INFO] Scaled ptjet_mjet_u_reco for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptjet_mjet_g_reco for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled response_matrix_u for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled response_matrix_g for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptjet_mjet_u_gen for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptjet_mjet_g_gen for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
[INFO] Scaled ptz_mz_reco for dataset pythia_UL18NanoAODv9_HT-400to600 by 10.688575 = 5.174 * 59830.0 / 32720.0
Done. time taken 22s
Output written to outputs/mass_pythia_local_2018_TEST.pkl with size 1.9 MB
[1] Saved: outputs/mass_pythia_local_2018_TEST.pkl
Number of group outputs: 1


In [56]:
for key in out:
    print(key)

ptjet_mjet_u_reco
ptjet_mjet_g_reco
response_matrix_u
response_matrix_g
ptjet_mjet_u_gen
ptjet_mjet_g_gen
ptz_mz_reco
sumw
nev
cutflow
